# unit_shap.csv 생성

LGBM fold 모델로 val unit별 SHAP Top10을 계산하여 `Dashboard/public/unit_shap.csv`에 저장합니다.

**전제조건**: Colab 환경에서 아래 변수를 직접 지정하거나 파이프라인에서 이미 `X_val`, `y_val`, `fold_models` 등이 메모리에 있는 상태여야 합니다.

## 실행 방법
1. Colab에서 모델 학습 노트북과 **같은 세션**에서 이 셀들을 실행 (또는 pkl 경로 지정)
2. `fold_models.pkl` + `X_val` (655개 피처, Column_0~654) 이 있어야 함
3. 실행 후 `unit_shap.csv` 다운로드 → Dashboard/public/ 에 복사

In [ ]:
# ── 환경 설정 ──────────────────────────────────────────────────────────────────
import os, sys

try:
    import google.colab
    IN_COLAB = True
    # 필요 패키지 설치
    os.system('pip install -q shap lightgbm')
except ImportError:
    IN_COLAB = False

print('Colab:', IN_COLAB)

In [ ]:
import pandas as pd
import numpy as np
import pickle
import shap
from pathlib import Path

# ── 경로 설정 ──────────────────────────────────────────────────────────────────
# Colab: /content/project/ 기준, 로컬: sk_하이닉스/ 기준
if IN_COLAB:
    ROOT = Path('/content/project')
else:
    ROOT = Path(__file__).parent.parent.parent if '__file__' in dir() else Path('.').resolve().parent.parent.parent

OUTPUT  = ROOT / '4_output'
FINAL   = OUTPUT / 'final'

# 모델 pkl (blend 기준 LGBM fold 모델)
MODEL_PKL = FINAL / 'reg_only' / 'lgbm' / 'fold_models.pkl'

# val 피처 (X_val — FS 후 Column_0~654 형태)
# 학습 파이프라인에서 직접 X_val을 주입하거나 아래 경로에서 로드
X_VAL_PATH = FINAL / 'reg_only' / 'lgbm' / 'X_val.pkl'  # 없으면 None

# val 메타 (ufs_serial 순서)
UNIT_VAL  = OUTPUT / 'unit_val.csv'

# 출력 경로
PUBLIC = ROOT / '6_분업' / 'Dashboard' / 'public'
PUBLIC.mkdir(parents=True, exist_ok=True)

print('ROOT   :', ROOT)
print('MODEL  :', MODEL_PKL.exists(), MODEL_PKL)
print('X_VAL  :', X_VAL_PATH.exists() if X_VAL_PATH else False)

## 방법 A: 학습 파이프라인에서 직접 주입

학습 노트북에서 이미 `X_val_fs`(FS 후 val 피처, shape=(8749, 655))와 `val_serials`(ufs_serial 리스트)가 있다면 아래 셀에 직접 할당합니다.

In [ ]:
# ── 방법 A: 파이프라인에서 직접 주입 ──────────────────────────────────────────
# 학습 노트북에서 아래 변수가 이미 정의되어 있으면 이 셀만 실행하면 됩니다

# X_val_fs: pd.DataFrame, shape=(8749, 655), columns=Column_0~Column_654
# val_serials: list or Series, len=8749, ufs_serial 순서 (X_val_fs와 동일 순서)

# 아래는 예시 — 실제로는 이미 메모리에 있어야 함
# X_val_fs = ...   (학습 노트북의 FS 후 val 피처 행렬)
# val_serials = ... (해당 unit의 ufs_serial)

# 정의 여부 확인
has_Xval = 'X_val_fs' in dir() and X_val_fs is not None
print('X_val_fs 정의됨:', has_Xval)
if has_Xval:
    print('  shape:', X_val_fs.shape)
    print('  columns[:5]:', list(X_val_fs.columns[:5]))

## 방법 B: pkl에서 X_val 로드

`4_output/final/reg_only/lgbm/X_val.pkl`이 있으면 로드합니다 (학습 시 저장해 두어야 함).

In [ ]:
# ── 방법 B: pkl 로드 ───────────────────────────────────────────────────────────
if not has_Xval:
    if X_VAL_PATH.exists():
        with open(X_VAL_PATH, 'rb') as f:
            X_val_fs = pickle.load(f)
        print('X_val.pkl 로드 완료:', X_val_fs.shape)
        has_Xval = True
    else:
        print('X_val.pkl 없음 → 방법 C(unit_val.csv fallback) 사용')

## 방법 C: unit_val.csv fallback (피처 이름 불일치 시 자동 매핑)

모델 피처 `Column_0~654`와 `unit_val.csv` 피처 `X0_mean~` 간 불일치를 처리합니다.
학습 코드에서 `feature_selected_indices`를 저장해 두었다면 자동 매핑 가능합니다.

In [ ]:
# ── 방법 C: 인덱스 매핑으로 피처 이름 복원 ────────────────────────────────────
# 학습 파이프라인에서 FS 후 컬럼 순서를 저장했다면:
# selected_cols_path = FINAL / 'reg_only' / 'lgbm' / 'selected_cols.pkl'

selected_cols_path = FINAL / 'reg_only' / 'lgbm' / 'selected_cols.pkl'

if not has_Xval and selected_cols_path.exists():
    with open(selected_cols_path, 'rb') as f:
        selected_cols = pickle.load(f)  # list of original column names
    
    unit_val = pd.read_csv(UNIT_VAL)
    unit_val.columns = [c.lower() if c != 'ufs_serial' else c for c in unit_val.columns]
    
    X_val_fs = unit_val[selected_cols].fillna(0)
    # Column_N 이름으로 rename
    X_val_fs.columns = [f'Column_{i}' for i in range(len(selected_cols))]
    val_serials = unit_val['ufs_serial']
    has_Xval = True
    print('방법 C 성공: selected_cols.pkl 매핑 완료, shape:', X_val_fs.shape)
elif not has_Xval:
    print('⚠ 방법 A/B/C 모두 실패')
    print('해결책: 학습 노트북에 아래 코드를 추가하여 X_val을 저장하세요:')
    print()
    print('  import pickle')
    print('  with open("4_output/final/reg_only/lgbm/X_val.pkl", "wb") as f:')
    print('      pickle.dump(X_val_fs, f)  # FS 후 val 피처 행렬')
    print('  # ufs_serial 순서도 함께 저장')
    print('  val_meta[["ufs_serial"]].to_csv("4_output/final/reg_only/lgbm/val_serials.csv", index=False)')

In [ ]:
# ── ufs_serial 로드 ────────────────────────────────────────────────────────────
val_serials_path = FINAL / 'reg_only' / 'lgbm' / 'val_serials.csv'

if 'val_serials' not in dir() or val_serials is None:
    if val_serials_path.exists():
        val_serials = pd.read_csv(val_serials_path)['ufs_serial']
        print('val_serials.csv 로드:', len(val_serials))
    else:
        # unit_val.csv에서 순서대로 추출 (X_val_fs와 동일 순서 가정)
        unit_val_meta = pd.read_csv(UNIT_VAL, usecols=['ufs_serial'])
        val_serials = unit_val_meta['ufs_serial']
        print('unit_val.csv ufs_serial 사용:', len(val_serials))

print('val_serials:', len(val_serials), 'units')

In [ ]:
# ── SHAP 계산 ─────────────────────────────────────────────────────────────────
if not has_Xval:
    print('⚠ X_val 없음 — 위 셀을 먼저 해결하세요')
else:
    # 모델 로드
    print('모델 로드 중...')
    with open(MODEL_PKL, 'rb') as f:
        ckpt = pickle.load(f)
    
    fold_list = ckpt['fold_models'] if isinstance(ckpt, dict) else ckpt
    model = fold_list[0]  # 첫 번째 fold 모델
    print(f'모델 로드 완료: {type(model).__name__}, 피처 수: {model.n_features_in_}')
    
    # X_val_fs 피처 수 확인
    n_model = model.n_features_in_
    n_val   = X_val_fs.shape[1]
    print(f'모델 피처: {n_model}, X_val 피처: {n_val}')
    
    if n_model != n_val:
        print(f'⚠ 피처 수 불일치! 모델({n_model}) ≠ X_val({n_val})')
        print('X_val_fs를 Column_0~Column_{n_model-1}으로 맞춰야 합니다.')
    else:
        print('피처 수 일치 — SHAP 계산 시작...')
        
        # SHAP 계산 (TreeExplainer)
        explainer = shap.TreeExplainer(model)
        
        # 대용량인 경우 배치로 계산
        BATCH = 1000
        n = len(X_val_fs)
        shap_batches = []
        for i in range(0, n, BATCH):
            batch = X_val_fs.iloc[i:i+BATCH]
            sv = explainer.shap_values(batch)
            shap_batches.append(sv)
            print(f'  배치 {i//BATCH + 1}/{(n + BATCH - 1)//BATCH} 완료')
        
        shap_values = np.vstack(shap_batches)  # (n_unit, n_feat)
        print(f'SHAP 계산 완료: {shap_values.shape}')

In [ ]:
# ── unit_shap.csv 저장 ────────────────────────────────────────────────────────
if has_Xval and 'shap_values' in dir():
    feat_names = list(X_val_fs.columns)
    
    shap_df = pd.DataFrame(shap_values, columns=feat_names)
    shap_df.insert(0, 'ufs_serial', val_serials.values)
    
    # wide → long
    shap_long = shap_df.melt(id_vars='ufs_serial', var_name='feature', value_name='shap_value')
    shap_long['abs_shap'] = shap_long['shap_value'].abs()
    
    # unit별 |shap| Top 10만 저장
    shap_top = (
        shap_long
        .sort_values(['ufs_serial', 'abs_shap'], ascending=[True, False])
        .groupby('ufs_serial')
        .head(10)
        .drop(columns='abs_shap')
        .reset_index(drop=True)
    )
    
    out_path = PUBLIC / 'unit_shap.csv'
    shap_top.to_csv(out_path, index=False)
    print(f'저장 완료: {out_path}')
    print(f'  행수: {len(shap_top):,}  (val {len(X_val_fs):,} units × Top10)')
    print()
    print('샘플:')
    display(shap_top.head(20))
else:
    print('⚠ shap_values 없음 — 위 셀을 먼저 실행하세요')

## 학습 노트북에 추가할 저장 코드

학습 노트북(`3_modeling/baseline.ipynb` 등)에서 FS 후 val 피처를 아래와 같이 저장해 두면 다음 실행 시 방법 B로 바로 사용 가능합니다.

```python
import pickle
from pathlib import Path

FINAL = Path('4_output/final/reg_only/lgbm')
FINAL.mkdir(parents=True, exist_ok=True)

# FS 후 val 피처 저장 (Column_0~654 컬럼명)
with open(FINAL / 'X_val.pkl', 'wb') as f:
    pickle.dump(X_val_fs, f)

# val ufs_serial 순서 저장
val_meta[['ufs_serial']].to_csv(FINAL / 'val_serials.csv', index=False)

print('저장 완료')
```

저장 후 Colab에서 이 노트북을 실행하면 방법 B가 자동으로 X_val을 로드합니다.